# 04 - Buyer Segmentation (K-means)

Groups European public buyers into procurement personas based on how they buy.

Input: workspace.gold.buyer_profiles (one row per buyer)
Output: cluster_id written back to buyer_profiles + a cluster profile summary table

Owner: Sebastiao (S4)
Model: K-means clustering, k selected by silhouette score

In [0]:
# Cell 1 - Imports and config
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import mlflow

SOURCE_TABLE = "workspace.gold.buyer_profiles"
OUTPUT_TABLE = "workspace.gold.buyer_profiles_clustered"
PROFILE_TABLE = "workspace.gold.cluster_profiles"

# numeric features used to define the clusters
FEATURE_COLS = [
    "total_contracts",
    "total_awarded_value_eur",
    "avg_award_value_eur",
    "distinct_cpv_codes",
    "single_bidder_rate",
    "cross_border_rate",
]

## Cell 2 - Load and inspect the data

Check how many buyers we have and what the features look like before clustering.
We need at least 500 buyers for clustering to be meaningful.

In [0]:
# Cell 2 - Load and inspect
df = spark.table(SOURCE_TABLE)

print("Total buyers:", df.count())
print("\nSchema:")
df.printSchema()

print("\nFeature summary:")
df.select(FEATURE_COLS).describe().show()

print("\nNull check per feature:")
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in FEATURE_COLS
]).show()

## Cell 3 - Clean the data

Drop buyers with null features (cannot cluster a row with missing values).
K-means is sensitive to extreme outliers, so we also cap the very top values
to stop a handful of mega-buyers from distorting the clusters.

In [0]:
# Cell 3 - Clean
df_clean = df.dropna(subset=FEATURE_COLS)
print("Buyers after dropping nulls:", df_clean.count())

# cap outliers at the 99th percentile for the two long-tailed value columns
for col in ["total_awarded_value_eur", "avg_award_value_eur"]:
    p99 = df_clean.approxQuantile(col, [0.99], 0.01)[0]
    df_clean = df_clean.withColumn(
        col,
        F.when(F.col(col) > p99, p99).otherwise(F.col(col))
    )
    print(f"Capped {col} at {p99:,.0f}")

## Cell 4 - Build the feature pipeline

Two steps:
1. VectorAssembler combines the 5 feature columns into one vector.
2. StandardScaler puts every feature on the same scale, so a feature measured
   in millions (spend) does not dominate one measured 0 to 1 (single bidder rate).


In [0]:
# Cell 4 - Assemble and scale
assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="features_raw"
)
df_vec = assembler.transform(df_clean)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)
scaler_model = scaler.fit(df_vec)
df_scaled = scaler_model.transform(df_vec)

print("Feature vector ready.")
df_scaled.select("buyer_name", "features").show(5, truncate=False)

## Cell 5 - Find the best k

Try k from 3 to 6. For each, fit K-means and measure the silhouette score
(how well separated the clusters are, higher is better, range -1 to 1).
We log every run to MLflow so the experiments are tracked.

In [0]:
# Cell 5 - Test different k values
mlflow.set_experiment("/Users/sebastiao.clemente@student.ie.edu/buyer_segmentation")

evaluator = ClusteringEvaluator(
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

results = {}

for k in [3, 4, 5, 6]:
    with mlflow.start_run(run_name=f"kmeans_k{k}"):
        kmeans = KMeans(featuresCol="features", k=k, seed=42)
        model = kmeans.fit(df_scaled)
        preds = model.transform(df_scaled)

        score = evaluator.evaluate(preds)
        results[k] = score

        mlflow.log_param("k", k)
        mlflow.log_metric("silhouette", score)

        print(f"k={k}  silhouette={score:.4f}")

best_k = max(results, key=results.get)
print(f"\nBest k = {best_k} (silhouette {results[best_k]:.4f})")

## Cell 6 - Fit the final model

Use the best k from the previous step. Fit it, attach the cluster_id to every buyer,
and log the final model to MLflow.

In [0]:
# Cell 6 - Final model
import mlflow

# serverless needs a UC Volume path to save Spark ML models
dfs_tmp = "/Volumes/workspace/default/lakehouse/mlflow_tmp"

with mlflow.start_run(run_name=f"kmeans_final_k{best_k}"):
    final_kmeans = KMeans(featuresCol="features", k=best_k, seed=42)
    final_model = final_kmeans.fit(df_scaled)
    df_clustered = final_model.transform(df_scaled)

    final_score = evaluator.evaluate(df_clustered)
    mlflow.log_param("k", best_k)
    mlflow.log_metric("silhouette", final_score)
    mlflow.spark.log_model(final_model, "kmeans_model", dfs_tmpdir=dfs_tmp)

    print(f"Final model fitted with k={best_k}")
    print(f"Silhouette: {final_score:.4f}")

# rename the prediction column to cluster_id
df_clustered = df_clustered.withColumnRenamed("prediction", "cluster_id")

print("\nBuyers per cluster:")
df_clustered.groupBy("cluster_id").count().orderBy("cluster_id").show()

## Cell 7 - Describe each cluster

This is the business output. For each cluster, compute the average of every feature
plus the most common country and CPV division. This is what turns "cluster 0, 1, 2"
into named personas like "high spend IT specialists".

In [0]:
# Cell 7 - Cluster profiles
cluster_profiles = (
    df_clustered.groupBy("cluster_id")
    .agg(
        F.count("*").alias("num_buyers"),
        F.avg("total_contracts").alias("avg_contracts"),
        F.avg("total_awarded_value_eur").alias("avg_total_spend"),
        F.avg("avg_award_value_eur").alias("avg_contract_size"),
        F.avg("distinct_cpv_codes").alias("avg_cpv_diversity"),
        F.avg("single_bidder_rate").alias("avg_single_bidder_rate"),
        F.avg("cross_border_rate").alias("avg_cross_border_rate"),
    )
    .orderBy("cluster_id")
)

print("Cluster profiles:")
cluster_profiles.show(truncate=False)

# most common country per cluster
print("Top country per cluster:")
from pyspark.sql.window import Window
w = Window.partitionBy("cluster_id").orderBy(F.desc("count"))
(df_clustered.groupBy("cluster_id", "buyer_country")
    .count()
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("cluster_id", "buyer_country", "count")
    .orderBy("cluster_id")
    .show())

# most common CPV division per cluster
print("Top CPV division per cluster:")
w2 = Window.partitionBy("cluster_id").orderBy(F.desc("count"))
(df_clustered.groupBy("cluster_id", "top_cpv_division")
    .count()
    .withColumn("rn", F.row_number().over(w2))
    .filter(F.col("rn") == 1)
    .select("cluster_id", "top_cpv_division", "count")
    .orderBy("cluster_id")
    .show())

## Cell 8 - Write results back to Gold

Two tables:
1. buyer_profiles_clustered: every buyer with their cluster_id, for the dashboard.
2. cluster_profiles: the summary table above, for the slide deck and dashboard.

In [0]:
# Cell 8 - Write outputs
output_df = df_clustered.select(
    "buyer_name",
    "buyer_country",
    "buyer_type",
    "total_contracts",
    "total_awarded_value_eur",
    "avg_award_value_eur",
    "distinct_cpv_codes",
    "single_bidder_rate",
    "cross_border_rate",
    "top_cpv_division",
    "cluster_id",
)

(output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE))

print("Wrote", output_df.count(), "buyers to", OUTPUT_TABLE)

(cluster_profiles.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PROFILE_TABLE))

print("Wrote cluster profiles to", PROFILE_TABLE)

## Cell 9 - Name the personas

Look at the cluster profiles above and give each cluster a human label.
Fill in the names based on what the numbers tell you, then save the final
labelled table. This is the cell you edit by hand after seeing the results.

In [0]:
# Cell 9 - Manual labelling (edited after seeing Cell 7 output)
cluster_labels = {
    
    0: "Occasional construction buyers",
    1: "Single-bidder health buyers",
    2: "High-volume power buyers",
    3: "Mega-contract outlier",
    4: "National centralised agencies",
    5: "Cross-border buyers",
}


# build a small dataframe mapping cluster_id to label
labels_df = spark.createDataFrame(
    [(k, v) for k, v in cluster_labels.items()],
    ["cluster_id", "cluster_label"]
)

# join labels onto the profiles and the buyer table
final_profiles = cluster_profiles.join(labels_df, "cluster_id", "left")
final_profiles.show(truncate=False)

(final_profiles.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PROFILE_TABLE))

print("Final labelled profiles saved.")

In [0]:
display(spark.table("workspace.gold.cluster_profiles"))